#**GLiNER on Contracts**

The script extracts information such as contract topic, companies involved, contract value etc from pdf contracts, using GLiNER-powered named entity recognition. You can choose different LLMs that run at the heart of the GLiNER library.

This notebook is designed to be run in Google colab.
It is in an early experimental stage.

Upload some sample contracts in pdf format in a new folder called "Vertrag" in the "content" folder on Colab

You can define the information you want GLiNER to extract in the main code, by editing "labels" list of strings (and the information to be displayed under "contract_info" dictionary list)

You can get sample contracts e.g. from the CUAD project:
https://www.atticusprojectai.org/cuad/


In [10]:
!pip install gliner pdfplumber thefuzz
!pip install pillow==10.4.0 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 119.0 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 12.2.0
    Uninstalling pillow-12.2.0:
      Successfully uninstalled pillow-12.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 10.4.0 which is incompatible.


# **Imports and definition of model:**

In [4]:
# code for colab:
# Works 3.3., additionally with batch processing and removal of duplicate extracts

import os
import pdfplumber
import pandas as pd
from thefuzz import fuzz
from gliner import GLiNER

#choose one of the pretrained LLMs below or any other model
#MODEL = "knowledgator/gliner-multitask-large-v0.5" #gut! 1.7GB, tokenlength: 768
#MODEL = "urchade/gliner_large-v2.1" #450 MB; good
#MODEL = "gliner-community/gliner_large-v2.5" #1.8 GB, tokenlength: 1024
#MODEL = "urchade/gliner_medium-v2.1" #200 MB
MODEL = "urchade/gliner_multi-v2.1" #1.16GB, token: 384, multilingual
#MODEL = "knowledgator/gliner-multitask-v1.0"
#MODEL = "knowledgator/modern-gliner-bi-base-v1.0"

# Initialize GLiNER model
model = GLiNER.from_pretrained(MODEL)

model = model.to('cuda:0')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

You must upload some training contract PDFs into a newly created "Vertrag" folder your Colab File explorer. The code below loads the contract PDFs from this folder.

**# main code**

In [5]:
#with output file name

THRESHOLD = 0.3 #std: 0.5-0.7
FUZZY_TRESH = 70 #std: 70

def extract_text_from_pdf(pdf_path):
    """
    Extracts text from a PDF file.
    """
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() + "\n"
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {e}")
    return text


def extract_contract_info(text, contract_path):
    """
    Extracts structured information from contract text using GLiNER.
    """
    # Define entity labels of interest (customize based on your needs and GLiNER model capabilities)
    labels = ["Arbeitnehmer", "Arbeitgeber", "Money", "Contractor", "Person", "Company",
              "PartyA", "PartyB", "Subject of Contract", "Date"]

    # Use words_splitter to handle long texts
    token_generator = model.data_processor.words_splitter(text)
    tokens = [t[0] for t in token_generator]  # Extract only the words from the tuples

    # Process tokens in batches
    batch_size = 1024  # Adjust this based on your model's capabilities
    all_entities = []

    for i in range(0, len(tokens), batch_size):
        batch = tokens[i:i+batch_size]
        batch_text = " ".join(batch)

        # Predict entities for this batch
        batch_entities = model.predict_entities(batch_text, labels, threshold=THRESHOLD)
        all_entities.extend(batch_entities)

    # Structure the extracted information
    contract_info = {
        "ContractPath": contract_path,
        "Arbeitnehmer": [],
        "Arbeitgeber": [],
        "Subject of Contract": [],
        "Contractor": [],
        "Money": [],
        "Person": [],
        "Company": [],
        "Sum": [],
        "PartyA": [],
        "PartyB": [],
        "Date": []
    }

    for entity in all_entities:
        label = entity["label"]
        text = entity["text"]

        # Append values to respective lists
        if label in contract_info:
            contract_info[label].append(text)

    # Fuzzy deduplication for each label
    for key in contract_info:
        if isinstance(contract_info[key], list):
            contract_info[key] = fuzzy_deduplicate(contract_info[key])

    return contract_info


def fuzzy_deduplicate(items, threshold=FUZZY_TRESH):
    """
    Removes duplicates and near-duplicates from a list of strings using fuzzy string matching.

    Args:
        items (list): List of strings to deduplicate.
        threshold (int): Similarity threshold (0-100) for considering two strings as near-duplicates.

    Returns:
        list: Deduplicated list of strings.
    """
    deduplicated = []

    for item in items:
        is_duplicate = False
        for existing_item in deduplicated:
            similarity = fuzz.ratio(item, existing_item)
            if similarity >= threshold:  # If similarity is above the threshold, consider it a duplicate
                is_duplicate = True
                break
        if not is_duplicate:
            deduplicated.append(item)

    return deduplicated


def save_to_excel(data, MODEL, THRESHOLD, FUZZY_TRESH):
    """
    Saves the extracted contract information to an Excel file with dynamic filename.
    """

    MODEL = MODEL.replace('/', '--')

    output_filename = (
        f"contracts_data_{MODEL}_threshold{THRESHOLD}_fuzzy{FUZZY_TRESH}.xlsx"
    )

    df = pd.DataFrame(data)

    # Show DataFrame inline
    print("\n=== DataFrame Preview ===")
    display(df)              # Jupyter/Colab rich display

    # Optional: show dimensions
    print(f"\nRows: {len(df)}, Columns: {len(df.columns)}")

    print(f"\nWriting to: {output_filename}")

    try:
        df.to_excel(output_filename, index=False)
        print(f"Data saved to {output_filename}")
    except Exception as e:
        print(f"Error saving to Excel: {e}")


# Set the directory containing the contract PDFs
contracts_directory = "/content/Vertrag"  # Replace with the actual path

# Change working directory to the contracts directory
os.chdir(contracts_directory)

# List files in the directory
files = os.listdir()
print("Files in the directory:", files)

# Process PDF files and extract information
all_contracts_info = []
for filename in os.listdir(contracts_directory):
    if filename.lower().endswith(('.pdf', '.PDF')):
        pdf_path = os.path.join(contracts_directory, filename)
        print(f"Processing {pdf_path}...")

        text = extract_text_from_pdf(pdf_path)

        if text:
            contract_info = extract_contract_info(text, pdf_path)
            all_contracts_info.append(contract_info)

# Save the extracted data to an Excel file with dynamic filename
if all_contracts_info:
    save_to_excel(all_contracts_info, MODEL, THRESHOLD, FUZZY_TRESH)
else:
    print("No contract information extracted.")


Files in the directory: ['ScansourceInc_20190509_10-Q_EX-10.2_11661422_EX-10.2_Distributor Agreement.pdf', 'StaarSurgicalCompany_20180801_10-Q_EX-10.37_11289449_EX-10.37_Distributor Agreement.pdf', 'PrecheckHealthServicesInc_20200320_8-K_EX-99.2_12070169_EX-99.2_Distributor Agreement.pdf', 'ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_Distributor Agreement2.pdf', 'GentechHoldingsInc_20190808_1-A_EX1A-6 MAT CTRCT_11776814_EX1A-6 MAT CTRCT_Distributor Agreement.pdf', 'InnerscopeHearingTechnologiesInc_20181109_8-K_EX-10.6_11419704_EX-10.6_Distributor Agreement.pdf', 'ScansourceInc_20190822_10-K_EX-10.39_11793959_EX-10.39_Distributor Agreement.pdf', 'SmartRxSystemsInc_20180914_1-A_EX1A-6 MAT CTRCT_11351705_EX1A-6 MAT CTRCT_Distributor Agreement.pdf', 'FuseMedicalInc_20190321_10-K_EX-10.43_11575454_EX-10.43_Distributor Agreement.pdf', 'ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_Distributor Agreement1.pdf', 'ZogenixInc_20190509_10-Q_EX-10.2_11663313_EX-10.2_Distributor 

/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 1024 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 807 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Processing /content/Vertrag/StaarSurgicalCompany_20180801_10-Q_EX-10.37_11289449_EX-10.37_Distributor Agreement.pdf...


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 980 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Processing /content/Vertrag/PrecheckHealthServicesInc_20200320_8-K_EX-99.2_12070169_EX-99.2_Distributor Agreement.pdf...
Processing /content/Vertrag/ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_Distributor Agreement2.pdf...


...

/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 606 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Processing /content/Vertrag/ImineCorp_20180725_S-1_EX-10.5_11275970_EX-10.5_Distributor Agreement.pdf...
Processing /content/Vertrag/WaterNowInc_20191120_10-Q_EX-10.12_11900227_EX-10.12_Distributor Agreement.pdf...

=== DataFrame Preview ===


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 740 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


,ContractPath,Arbeitnehmer,Arbeitgeber,Subject of Contract,Contractor,Money,Person,Company,Sum,PartyA,PartyB,Date
0,/content/Vertrag/ScansourceInc_20190509_10-Q_E...,[],[],"[Distribution Agreement, purchase orders]",[],[],"[Alex Castaneda, Brenda McCurry]","[ZEBRA, Zebra Technologies International , LLC...",[],[Xplore],[],"[4th day of February 2019, Estimated Shipping ..."
1,/content/Vertrag/StaarSurgicalCompany_20180801...,[employee],[STAAR],"[DISTRIBUTORSHIP AGREEMENT, Products, data sub...","[subcontractors, Distributor]","[standard repair charges, shipping expenses, m...",[],"[STAAR SURGICAL AG, STAAR, Distributor, Compan...",[],"[Distributor, terminating party, parties]",[STAAR],"[____________, Effective Date, March 10, June ..."
2,/content/Vertrag/PrecheckHealthServicesInc_202...,[],[],[],[],[],"[Cameron Gundry, Justin Anderson]","[Co-Diagnostics , Inc ., Principal, PreCheck H...",[],[],[],"[19th day of March , 2020, 3 / 20 / 2020]"
3,/content/Vertrag/ScansourceInc_20190822_10-K_E...,[],[],[Software License Agreement],[],"[FULL REFUND, license fees, LOST REVENUE, PROFIT]","[CUSTOMER, Customer]","[CISCO, Cisco, Cisco Systems , Inc .]",[],[],[],[]
4,/content/Vertrag/GentechHoldingsInc_20190808_1...,[employees],[],"[DISTRIBUTOR AGREEMENT, business and marketing...",[],"[Payment, remaining balance, finances]",[Customers],"[B & C General Warehouse Corporation LLC, Comp...",[],[Parties],"[Disclosing Party, Recipient Party, Indemnifyi...","[August 2019, anniversary date, 8 / 8 / 2019, ..."
5,/content/Vertrag/InnerscopeHearingTechnologies...,[employees],[],[Products],[],"[U . S . Dollars, binding arbitration, reasona...","[licensed health care professional, officers, ...","[ERCHONIA CORPORATION, DISTRIBUTOR, Erchonia C...",[],"[Distributor, parties]","[third party, Distributor]","[delivery date, 11 / 9 / 2018]"
6,/content/Vertrag/ScansourceInc_20190822_10-K_E...,[],[],[NONEXCLUSIVE VALUE ADDED DISTRIBUTOR AGREEMEN...,[],"[payment, money]","[Government officials, Any person, S . K . Ver...","[Cisco Systems , Inc ., Cisco, ScanSource , In...",[],[Distributor],[Distributor],"[Amendment Effective Date, January 22 , 2007, ..."
7,/content/Vertrag/SmartRxSystemsInc_20180914_1-...,[],"[A3 Development Group , LLC]","[Class A Series Agreement, EXCLUSIVE DISTRIBUT...",[],"[management fee, Net Income, distributable cas...","[arbitrator, Sandeep Mathow, Tracy W . Gunter]","[SMART RX SYSTEMS , INC ., Company, A3 DEVELOP...",[],"[Distributor, Customers in the Territory, Cust...",[],"[17th day of May , 2017, date of the transacti..."
8,/content/Vertrag/FuseMedicalInc_20190321_10-K_...,[],"[Signature Orthopaedics Pty Ltd, CPM Medical LLC]","[Products, Relevant Claim]",[the Distributor],"[monies, AU $ 10 million, assets, $ 610, $ 450...","[buying agent, suitably qualified personnel, p...","[Signature Orthopaedics Pty Ltd, CPM Medical C...",[],"[Supplier, Distributor, the other party, Permi...","[Supplier, the Distributor, the other party]","[29 / 3 / 18, Commencement Date, 3 / 21 / 2019..."
9,/content/Vertrag/ScansourceInc_20190822_10-K_E...,"[employees, EMPLOYEES]",[],[NONEXCLUSIVE VALUE ADDED DISTRIBUTOR AGREEMEN...,[],"[appropriate amount, credit, revenue, certifie...","[Reseller, authorized representative, Customer...","[CISCO SYSTEMS , INC ., ScanSource , Inc ., Ci...",[],"[parties hereto, Approved Source, Distributor,...","[Distributor, Integrator]","[Effective Date, 8 / 22 / 2019, Effective Date..."



Rows: 13, Columns: 12

Writing to: contracts_data_urchade--gliner_multi-v2.1_threshold0.3_fuzzy70.xlsx
Data saved to contracts_data_urchade--gliner_multi-v2.1_threshold0.3_fuzzy70.xlsx
